[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ricalanis/openenv-hackaton/blob/main/training/train_cleaning.ipynb)

# DataSage Stage 1: Cleaning GRPO Training

Pre-builds dataset with real environment observations (no `rollout_func`).
Reward functions replay the env with stored seeds for context-matched evaluation.

**Requirements:** GPU (A100/H100), `HF_TOKEN`, `WANDB_API_KEY` env vars.

In [ ]:
# Setup: install dependencies
!pip install -q unsloth trl datasets wandb pydantic requests
!pip install -q openenv-core

In [ ]:
# Setup: clone repo and configure paths
import os, sys

REPO_URL = "https://github.com/ricalanis/openenv-hackaton.git"  # UPDATE THIS
REPO_DIR = "/content/datasage"

if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ:
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    sys.path.insert(0, REPO_DIR)
else:
    # Running locally from training/ dir
    project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if project_root not in sys.path:
        sys.path.insert(0, project_root)

print(f"Working directory: {os.getcwd()}")

In [ ]:
# Load API keys from Colab Secrets or set manually
import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    print("Loaded keys from Colab Secrets")
except Exception:
    pass  # Fill manually below if not using Colab Secrets

# Uncomment and fill if not using Colab Secrets:
# os.environ["HF_TOKEN"] = "hf_..."
# os.environ["WANDB_API_KEY"] = "wandb_..."

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in Colab Secrets or above"
print("Keys loaded.")

In [ ]:
# Imports
import random
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

from training.shared.config import (
    BASE_MODEL, HF_MODEL_REPOS, SPACE_URLS, TRAINING_CONFIGS, WANDB_PROJECT,
)
from training.shared.parsers import parse_cleaning_action
from training.shared.rewards import cleaning_json_format_reward
from environments.cleaning.client import CleaningEnv
from environments.cleaning.models import CleaningAction

ENV_URL = SPACE_URLS["cleaning"]
STAGE_CONFIG = TRAINING_CONFIGS["cleaning"]
print(f"Environment URL: {ENV_URL}")
print(f"Config: {STAGE_CONFIG}")

In [ ]:
# Model loading via Unsloth
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {BASE_MODEL}")

In [ ]:
# System prompt and task descriptions
SYSTEM_PROMPT = """\
You are a data quality agent. You clean enterprise datasets across multiple \
domains (HR, Sales, Project Management, IT Operations).

Analyze the data quality report and data preview below, then respond with a \
JSON cleaning action:
{"operation": "<op>", "column": "<col>", "value": "<val>", "params": {}}

Available operations:
- fill_null: Fill null values (value can be "median", "mode", or a specific value)
- fix_type: Fix type mismatches in a column (cast to proper type)
- remove_duplicate: Remove duplicate rows
- standardize: Standardize column values (strip whitespace, normalize case)
- trim: Trim whitespace from column values
- correct_typo: Correct typos in categorical values

Identify the most impactful issue and act."""

TASK_DESCRIPTIONS = [
    "Clean the data to maximize the data quality score.",
    "Fix the most impactful data quality issue in this dataset.",
    "Analyze the DQ report and apply the best cleaning operation.",
    "This data has quality issues. Identify and fix the worst one.",
    "Improve the data quality score as much as possible with one operation.",
    "Look at the null counts, type issues, and duplicates. Fix the biggest problem.",
    "The data quality score is low. Apply the cleaning operation with the highest impact.",
    "Examine the data preview and quality report. What single operation improves quality most?",
]

In [ ]:
# Dataset: pre-built with real environment observations
random.seed(42)
SEEDS = [42 + i for i in range(64)]
prompts = []

print("Building dataset with real environment observations...")
for i, seed in enumerate(SEEDS):
    task_desc = random.choice(TASK_DESCRIPTIONS)
    with CleaningEnv(base_url=ENV_URL) as client:
        result = client.reset_with_seed(seed=seed)
        obs = result.observation

    prompts.append([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            f"Domain: {obs.domain}\n\n"
            f"Data Quality Report:\n{obs.dq_report}\n"
            f"DQ Score: {obs.dq_score:.4f}\n\n"
            f"Columns:\n{obs.columns_info}\n\n"
            f"Data Preview:\n{obs.data_preview}\n\n"
            f"Task: {task_desc}"
        )},
    ])
    if (i + 1) % 16 == 0:
        print(f"  Built {i + 1}/64 examples")

dataset = Dataset.from_dict({
    "prompt": prompts,
    "seed": SEEDS,
})
print(f"Dataset size: {len(dataset)}")

In [ ]:
# Environment reward function (calls env directly with stored seeds)
def cleaning_env_reward(completions: list[str], **kwargs) -> list[float]:
    """Primary reward: replay env with stored seed, step with parsed action."""
    seeds = kwargs.get("seed", [0] * len(completions))
    rewards = []
    for text, seed in zip(completions, seeds):
        try:
            action_dict = parse_cleaning_action(text)
            action = CleaningAction(
                operation=action_dict.get("operation", "fill_null"),
                column=action_dict.get("column", ""),
                value=action_dict.get("value"),
                params=action_dict.get("params", {}),
            )
            with CleaningEnv(base_url=ENV_URL) as client:
                client.reset_with_seed(seed=int(seed))
                result = client.step(action)
                rewards.append(float(result.reward or 0.0))
        except Exception as e:
            print(f"Env error: {e}")
            rewards.append(0.0)
    return rewards

In [ ]:
# GRPO training config
training_args = GRPOConfig(
    output_dir="./outputs/cleaning-grpo",
    learning_rate=STAGE_CONFIG["learning_rate"],
    num_train_epochs=STAGE_CONFIG["num_train_epochs"],
    per_device_train_batch_size=STAGE_CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=STAGE_CONFIG["gradient_accumulation_steps"],
    num_generations=STAGE_CONFIG["num_generations"],
    max_completion_length=STAGE_CONFIG["max_completion_length"],
    max_prompt_length=STAGE_CONFIG["max_prompt_length"],
    logging_steps=1, save_steps=50, bf16=True,
    use_vllm=True, vllm_mode="colocate",
    report_to="wandb", run_name="datasage-cleaning-grpo-v2",
)
os.environ.setdefault("WANDB_PROJECT", WANDB_PROJECT)

In [ ]:
# Create trainer and train
trainer = GRPOTrainer(
    model=model, processing_class=tokenizer, args=training_args,
    train_dataset=dataset,
    reward_funcs=[cleaning_env_reward, cleaning_json_format_reward],
)
print("Starting Stage 1 (Cleaning) GRPO training...")
trainer.train()

In [ ]:
# Save and push to Hub
output_dir = "./outputs/cleaning-grpo-final"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model saved to {output_dir}")

hf_repo = HF_MODEL_REPOS["cleaning"]
print(f"Pushing to Hub: {hf_repo}")
trainer.push_to_hub(hf_repo)
print(f"Model pushed to https://huggingface.co/{hf_repo}")